## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 05: Robustness, Substitution and Market Quality**

---

**Scope:** In-time placebo, Bank-Nifty-only before/after, window sensitivity; substitution accounting with a sensitivity range for organic growth; participant mix; futures market-quality spillovers. Reproduce: `python -m scripts.substitution_placebo_audit`, `python -m scripts.cross_exchange_did`, `python -m scripts.market_quality_did`.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, F&O Bhavcopy archive (NOTE: the rule-based expiry calendar it builds is NOT used for inference)
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical features (true-calendar expiry features) and the true-expiry index panel
3. **03_weekly_options_discontinuation_identification.ipynb** -- true expiry dates per index and a calendar audit
4. **04_primary_diff_in_diff_analysis.ipynb** -- corrected symbol-month DiD (turnover, expiry-day turnover, expiry-day share)
5. **05_robustness_checks.ipynb** -- placebo, window sensitivity, substitution range, participant mix, futures market quality
6. **06_expiry_volume_event_study.ipynb** -- expiry-day concentration for six NSE/BSE indices
7. **07_walk_forward_predictability.ipynb** -- secondary: leakage-free predictability test on the true calendar (null)
8. **08_interrupted_time_series.ipynb** -- secondary: settlement-weekday ITS plus clean weekday-only contrasts (null)
9. **09_figures_for_paper.ipynb** -- displays the corrected figures
10. **10_option_microstructure_and_placebo.ipynb** -- option-level outcomes, stock-F&O placebo, dose-response

**Current location:** **05_robustness_checks.ipynb** <- *you are here*


In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\Research-IPR\Research_BN_N50\banknifty-expiry-volatility-ml


### 1. Placebo, BN-only and window sensitivity (NSE)

In [2]:
r = json.load(open(config.RESULTS_DIR / 'corrected_robustness.json'))
keys = ['in_time_placebo', 'bn_only_before_after', 'window_sensitivity']
print(json.dumps({k: r[k] for k in keys}, indent=1))

{
 "in_time_placebo": {
  "lt": {
   "true": -2.8210112182722216,
   "placebo_mean": -0.04126436000806456,
   "placebo_min": -0.40572222167121375,
   "placebo_max": 0.2599233510397214,
   "n_placebos": 31,
   "rand_p_two_sided": 0.0
  },
  "le": {
   "true": -1.4207184681123424,
   "placebo_mean": -0.039393601838801556,
   "placebo_min": -0.5154102235405684,
   "placebo_max": 0.30898150799322854,
   "n_placebos": 31,
   "rand_p_two_sided": 0.0
  },
  "ls": {
   "true": 1.4002927501598814,
   "placebo_mean": 0.0018707581692627357,
   "placebo_min": -0.11449485959164901,
   "placebo_max": 0.11560403113177023,
   "n_placebos": 31,
   "rand_p_two_sided": 0.0
  }
 },
 "bn_only_before_after": {
  "lt": {
   "coef": -3.4701829498590504,
   "se": 0.35330075910116404
  },
  "le": {
   "coef": -2.312454407506874,
   "se": 0.27835371768018324
  },
  "ls": {
   "coef": 1.1577285423521722,
   "se": 0.09111305185710199
  }
 },
 "window_sensitivity": {
  "2023-01..2026-10": {
   "lt": -2.102105911992

### 2. Substitution (six indices)
The raw share is ~99%, but both controls were growing fast before the reform, so the share depends on the assumed organic growth. Treat it as a range, not a point estimate.

In [3]:
x = json.load(open(config.RESULTS_DIR / 'cross_exchange_did.json'))
print(json.dumps(x['substitution'], indent=1))
print(json.dumps(x['participants'], indent=1))

{
 "treated_loss_per_month": 4618442366766176.0,
 "controls_gain_naive": 4573674847952138.0,
 "years_between_period_midpoints": 1.7494866529774127,
 "substitution_share_naive": 0.9903067927974634,
 "substitution_share_by_assumed_organic_growth": {
  "organic_growth_0pct_per_year": 0.9903067927974634,
  "organic_growth_25pct_per_year": 0.6302251954161278,
  "organic_growth_50pct_per_year": 0.21164905948058454,
  "organic_growth_100pct_per_year": -0.7909710845499941
 },
 "total_6index_change_pct_naive": -0.5259563040297399,
 "sensex_gain_naive": 2571025806048213.0,
 "nifty_gain_naive": 2002649041903925.0
}
{
 "pre (2024-01..09)": {
  "Client": 0.34095615443758637,
  "Pro": 0.586579047448025,
  "FII": 0.07242774236002435,
  "DII": 3.705575436420378e-05,
  "opt_total_long_mean": 502520722.86885244
 },
 "post (2025-01..2026-09)": {
  "Client": 0.36444448880908414,
  "Pro": 0.5629994970728028,
  "FII": 0.0725188632433546,
  "DII": 3.71508747585694e-05,
  "opt_total_long_mean": 140018233.1435

### 3. Market quality (near-month index futures; spillover outcomes, not option spreads)

In [4]:
q = json.load(open(config.RESULTS_DIR / 'market_quality_did.json'))
within = {k: {o: (round(v['coef'], 4), round(v['p'], 3)) for o, v in q[k].items()} for k in q if ': ' in k}
print(pd.DataFrame(within).T)
print(pd.DataFrame(q['pooled_twfe']).T)

                                 l_amihud         cs_spread              l_hl  \
NSE: BANKNIFTY vs NIFTY  (-0.0132, 0.957)  (-0.0006, 0.166)  (-0.1705, 0.008)   
BSE: BANKEX vs SENSEX     (1.0625, 0.155)   (0.0003, 0.582)  (-0.0696, 0.755)   

                          zero_days  
NSE: BANKNIFTY vs NIFTY  (0.0, nan)  
BSE: BANKEX vs SENSEX    (0.0, nan)  
               coef  randomization_p  n_assign
l_amihud   1.362390         0.266667      15.0
cs_spread  0.000132         0.866667      15.0
l_hl      -0.164339         0.600000      15.0
zero_days  0.000000         1.000000      15.0
